# Data Extraction Pipeline

In [2]:
pip install numpy

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 79.8 MB/s  0:00:006m0:00:01
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install astropy

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 63.4 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.7/738.7 kB 14.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [astropy]m2/3 [astropy]iers-data]
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install h5py

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 27.2 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
import astropy.io
from astropy.io import fits
import h5py

Input the the path to the files as a string here:

In [2]:
filename = "201045030/outputFile"

In [3]:
obj = fits.open(filename + '.pi')
arf = fits.open(filename + '.arf')
rmf = fits.open(filename + '.rmf')

In [4]:
def construct_response(fits_file_dir, fits_arf_dir, min_val = 1.e-6, nustar = False, hitomi = False, acis = False, ROSAT = False):

    with fits.open(fits_file_dir) as rmf:

        # Calculate the detector response, using the arf and rmf                                                                                                                                                                                                   
        # This maps from a series of input channels (energies) to output ones,                                                                                                                                                                                     
        # accounting for the effective area. This is a square matrix for the mos                                                                                                                                                                                   
        # camera, but not for the pn                                                                                                                                                                                                                               

        # Get the input and output energy arrays - for pn not uniformly spaced                                                                                                                                                                                     
        cin_min = rmf['MATRIX'].data['ENERG_LO'] # [keV]                                                                                                                                                                                                           
        cin_max = rmf['MATRIX'].data['ENERG_HI'] # [keV]                                                                                                                                                                                                           

        cout_min = rmf['EBOUNDS'].data['E_MIN'] # [keV]                                                                                                                                                                                                            
        cout_max = rmf['EBOUNDS'].data['E_MAX'] # [keV]                                                                                                                                                                                                            

        cout_de = cout_max - cout_min # [keV]                                                                                                                                                                                                                      

    if not ROSAT:
        with fits.open(fits_arf_dir) as arf:
            # Extract the effective area as a function of input energy                                                                                                                                                                                             
            # NB: this is vignetting corrected                                                                                                                                                                                                                     
            effA = arf[1].data['SPECRESP'] # [cm^2]                                                                                                                                                                                                                
    else:
        effA = np.ones(len(cin_min))

    # Extract the rmf matrix, which gives the probability of an input X-ray with                                                                                                                                                                                   
    # true energy in an input channel is reconstructed into a given output channel                                                                                                                                                                                 
    # At the same time we also multiply in the effective area through, combining                                                                                                                                                                                   
    # to give the full detector response                                                                                                                                                                                                                           
    # NB: this matrix is stored in a sparse format, so we have to reconstruct it                                                                                                                                                                                   

    with fits.open(fits_file_dir) as rmf:

        # Check shape of F_CHAN, it and N_CHAN have different shape for pn than mos                                                                                                                                                                                
        # For mos, N_GRP is always 0 or 1, for pn it can be larger                                                                                                                                                                                                 
        # NuSTAR chooses one of these, because most generally it is just if there                                                                                                                                                                                  
        # are multiple groups.                                                                                                                                                                                                                                     
        pndat = 0
        if len(np.shape(rmf['MATRIX'].data['F_CHAN'])) == 2:
            pndat = 1

        det_res = np.zeros((len(cout_min),len(cin_min)))

        for i in range(len(cin_min)):

            # Reconstruct sparse matrix                                                                                                                                                                                                                            
            det_col = np.zeros(len(cout_min))
            jtot = 0
            for j in range(rmf['MATRIX'].data['N_GRP'][i]): # Number of groups                                                                                                                                                                                     
                # Account for different file structure in pn and mos                                                                                                                                                                                               
                if pndat or nustar or hitomi:
                    fc = rmf['MATRIX'].data['F_CHAN'][i][j] # First channel                                                                                                                                                                                        
                    sl = rmf['MATRIX'].data['N_CHAN'][i][j] # Channels in this group                                                                                                                                                                               
                elif acis:
                    fc = rmf['MATRIX'].data['F_CHAN'][i][j]-1 # First channel but it's 1-indexed                                                                                                                                                                   
                    sl = rmf['MATRIX'].data['N_CHAN'][i][j] # Channels in this group                                                                                                                                                                               
                else:
                    fc = rmf['MATRIX'].data['F_CHAN'][i] # First channel                                                                                                                                                                                           
                    sl = rmf['MATRIX'].data['N_CHAN'][i] # Channels in this group                                                                                                                                                                                  
                det_col[fc:fc+sl] = rmf['MATRIX'].data['MATRIX'][i][jtot:jtot+sl]
                jtot += sl
            # To help compress matrix, set all values < min_val to 0                                                                                                                                                                                               
            # Recall this is a pdf, so these entries have negligible impact                                                                                                                                                                                        
            # NB: rmfgen already has cut values < 1.e-6                                                                                                                                                                                                            
            tocut = np.where(det_col < min_val)
            det_col[tocut] = 0.

            # Save and account for effective area                                                                                                                                                                                                                  
            det_res[:,i] = det_col * effA[i] # [cm^2]                                                                                                                                                                                                              

    return cin_min, cin_max, cout_min, cout_max, det_res

In [5]:
cin_min, cin_max, cout_min, cout_max, det_res = construct_response("201045030/outputFile.rmf", "201045030/outputFile.arf", min_val = 1.e-6, nustar = False, hitomi = True, acis = False, ROSAT = False)

In [6]:
# Calculate the differential flux in each bin
# NB: this has units of [cts/s/keV/sr], which is often how X-ray results
# are plotted. The cm^2 is wrapped up in the detector response, and stored
# seprately
cout_de = cout_max - cout_min # [keV]
counts = obj['SPECTRUM'].data['COUNTS']
exp = obj['SPECTRUM'].header['EXPOSURE'] # [s]
roi_size = obj['SPECTRUM'].header['BACKSCAL']*(0.05*1./60./60.*np.pi/180.)**2.
flux = counts/cout_de/exp/roi_size


# Write the output as an h5 file, compressing the detector response
out_file = filename + '_processed.h5'
h5f = h5py.File(out_file, 'w')
h5f.create_dataset('counts',data=counts)
h5f.create_dataset('flux',data=flux)
h5f.create_dataset('det_res',data=det_res,compression='gzip',compression_opts=9)
h5f.create_dataset('exp',data=exp)
h5f.create_dataset('roi_size',data=roi_size)
h5f.create_dataset('cin_min',data=cin_min)
h5f.create_dataset('cin_max',data=cin_max)
h5f.create_dataset('cout_min',data=cout_min)
h5f.create_dataset('cout_max',data=cout_max)
h5f.close()

In [ ]:
# # Calculate the detector response, using the arf and rmf
# # This maps from a series of input channels (energies) to output ones,
# # accounting for the effective area. This is a square matrix for the mos
# # camera, but not for the pn

# # Get the input and output energy arrays - for pn not uniformly spaced
# cin_min = rmf['MATRIX'].data['ENERG_LO'] # [keV]
# cin_max = rmf['MATRIX'].data['ENERG_HI'] # [keV]

# cout_min = rmf['EBOUNDS'].data['E_MIN'] # [keV]
# cout_max = rmf['EBOUNDS'].data['E_MAX'] # [keV]
# cout_de = cout_max - cout_min # [keV]

# print(cin_min)
# print(cin_max)
# print(cout_min)
# print(cout_max)
# print(cout_de)


In [ ]:
# # Extract the raw X-ray counts in each CCD channel. Number of channels is different if mos or pn camera
# # Each channel is associated with an energy, as extracted from the detector - response files below
# counts = obj['SPECTRUM'].data['COUNTS']

# print(counts)

# # Extract the exposure time for the entire observation
# # Not vignetting corrected
# exp = obj['SPECTRUM'].header['EXPOSURE'] # [s]

# print(exp)

# # Extract the size of the ROI from backscale
# # units are (0.05'')^2, so convert to sr
# roi_size = obj['SPECTRUM'].header['BACKSCAL']*(0.05*1./60./60.*np.pi/180.)**2.

# print(roi_size)

In [15]:
# Extract the effective area as a function of input energy
# NB: this is vignetting corrected
#effA = arf[1].data['SPECRESP'] # [cm^2]

# Extract the rmf matrix, which gives the probability of an input X-ray with
# true energy in an input channel is reconstructed into a given output channel
# At the same time we also multiply in the effective area through, combining
# to give the full detector response
# NB: this matrix is stored in a sparse format, so we have to reconstruct it

# Check shape of F_CHAN, it and N_CHAN have different shape for pn than mos
# For mos, N_GRP is always 0 or 1, for pn it can be larger
#pndat = 0
#if len(np.shape(rmf['MATRIX'].data['F_CHAN'])) == 2:
#    pndat = 1

#det_res = np.zeros((len(cout_min),len(cin_min)))

#print(pndat)
#print(det_res)


In [16]:
# for i in range(len(cin_min)):
    
#     # Reconstruct sparse matrix
#     det_col = np.zeros(len(cout_min))
#     jtot = 0
#     for j in range(rmf['MATRIX'].data['N_GRP'][i]): # Number of groups
#         # Account for different file strucutre in pn and mos
#         if pndat:
#             fc = rmf['MATRIX'].data['F_CHAN'][i][j] # First channel
#             sl = rmf['MATRIX'].data['N_CHAN'][i][j] # Channels in this group
#         else:
#             fc = rmf['MATRIX'].data['F_CHAN'][i] # First channel
#             sl = rmf['MATRIX'].data['N_CHAN'][i] # Channels in this group
        
#         #In at least one instance, we wind up running out of entries in our matrix. To avoid this being fatal, we just skip these entries
#         #if jtot + sl[0] > np.shape(rmf['MATRIX'].data['MATRIX'][i]):
#         #    continue
        
#         det_col[fc[0]:fc[0]+sl[0]] = rmf['MATRIX'].data['MATRIX'][i][jtot:(jtot+sl[0])]

#         jtot += sl[0]

#     # To help compress matrix, set all values < 1.e-5 to 0
#     # Recall this is a pdf, so these entries have negligible impact
#     tocut = np.where(det_col < 1.e-5)
#     det_col[tocut] = 0

#     # Save and account for effective area
#     det_res[:,i] = det_col * effA[i] # [cm^2]

ValueError: could not broadcast input array from shape (26,) into shape (58,)